<a href="https://colab.research.google.com/github/JohnThefive/Criador-de-personagem---Tormenta-20/blob/main/Experimento_com_LLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instala as bibliotecas necessárias
!pip install -q -U google-generativeai chromadb

Configuração e Leitura dos Dados

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# Recupera a chave de forma segura
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

#Instancia o modelo (Recomendo o 1.5 Flash para processar transcrições longas rapidamente)
model = genai.GenerativeModel('gemini-3.1-flash-lite')

arquivos_transcricao = [
    "Gravacao_rpg_9_04_parte_1.txt",
    "Gravacao_rpg_9_04_parte_2.txt",
    "Gravacao_rpg_9_04_parte_3.txt",
    "Gravacao_rpg_9_04_parte_4.txt"
]

transcricao_completa = ""

# Le e conecta todos os arquivos mantendo a estrutura de Speakers
print("Carregando e unificando arquivos com diarização...")
try:
    for i, nome_arq in enumerate(arquivos_transcricao, 1):
        with open(nome_arq, 'r', encoding='utf-8') as f:
            conteudo = f.read()
            transcricao_completa += f"\n--- PARTE {i} ---\n" + conteudo
    print("Sucesso! Todos os arquivos foram consolidados")
except FileNotFoundError as e:
    print(f"ERRO: Não encontrei o arquivo {e.filename}. Verifique se fez o upload correto na barra lateral.")

# Atualiza a variável que os prompts usam
transcricao = transcricao_completa


Carregando e unificando arquivos com diarização...
Sucesso! Todos os arquivos foram consolidados


Teste 0-Shot

In [ ]:
prompt_zero_shot = f"""

Atue com uma dupla persona para analisar a transcrição diarizada de uma sessão real de RPG de mesa (Tormenta 20):

PERSONA 1: Mestre de Jogo (MJ) veterano de Tormenta 20. Conhece profundamente o sistema de regras, tem elevada empatia pela sobrecarga cognitiva exigida para gerir a mesa e deteta facilmente quando os jogadores ficam confusos com as mecânicas ou quando o ritmo narrativo quebra devido a microgestão (cálculos, consulta de tabelas, estados).
PERSONA 2: Engenheiro de Requisitos e Investigador de Interação Pessoa-Máquina (IPM), focado na conceção (design) de ferramentas digitais de suporte para MJs.

A sua tarefa deve ser executada em dois passos obrigatórios:

PASSO 1 (Visão do Mestre): Analise a transcrição abaixo e identifique os "Pontos de Atrito" (Friction Points) da sessão. Procure por:
- Jogadores confusos com magias, testes de resistência ou regras de combate.
- Momentos de hesitação do Mestre para consultar notas ou fazer cálculos mentais (ex: pontos de vida, bónus acumulados, iniciativa).
- Quebras de imersão.
Para cada ponto, cite o excerto exato da transcrição que o comprova.

PASSO 2 (Visão do Engenheiro): Para cada "Ponto de Atrito" identificado, mude para a perspetiva de Engenheiro de Software e proponha:
- Requisito Funcional (RF): Uma funcionalidade de software específica que resolveria este problema (não sugira "técnicas de mestragem", proponha ferramentas computacionais puras).
- Justificação de Design: Explique como essa funcionalidade reduz a carga cognitiva do utilizador (Mestre) sem restringir a sua liberdade criativa.

Transcrição da Sessão:
{transcricao_completa}
"""

resposta_zero = model.generate_content(prompt_zero_shot)
print(resposta_zero.text)

Esta é uma análise técnica e sistêmica da sessão de RPG apresentada, dividida pelas duas personas solicitadas.

---

### PASSO 1: Visão do Mestre de Jogo (MJ)

Como veterano, percebo uma mesa com grande potencial criativo, mas que sofre com uma **crônica falta de fluidez mecânica**. A carga cognitiva está sendo desperdiçada em burocracia básica, o que impede a imersão total.

**Pontos de Atrito Identificados:**

1.  **Dúvidas constantes sobre a ficha e modificadores:** Os jogadores não dominam os seus bônus, resultando em pausas frequentes para cálculos básicos.
    *   *Excerto:* "15 e mais nada. Mais 2, 3, né? [...] Qual o nome desse? O Celso. O Celso." / "Minha luta... É 6. 8 mais 6, 14."
2.  **Confusão com regras de manobras e condições:** Há um debate constante sobre o que constitui uma ação, o que gera ataques de oportunidade e como funcionam condições simples.
    *   *Excerto:* "Eu vou tentar usar a corda pra segurar o pescoço dele... Pode ser? Tem que fazer uma manobra de agar

Teste One-Shot

In [ ]:
# ---  TESTE ONE-SHOT  ---
prompt_one_shot = f"""
Atue com uma dupla persona para analisar a transcrição diarizada de uma sessão real de RPG (Tormenta 20).
Persona 1: Mestre de Jogo veterano de Tormenta 20, com olhar crítico sobre regras e ritmo.
Persona 2: Engenheiro de Requisitos especialista em ferramentas de suporte computacional.

Para garantir o rigor científico da extração de requisitos, siga estritamente o formato de análise demonstrado no exemplo abaixo.

--- INÍCIO DO EXEMPLO ---
Entrada (Excerto):
[Mestre]: "O Minotauro vai atacar o clérigo com a machadada. Deixa ver... ele tem mais 8 de ataque, mas está flanqueado e... espera, ele sofreu aquele feitiço de lentidão, certo?"
[Jogador - Mago]: "Sim, falhou no teste de Vontade no meu turno."
[Mestre]: "Ok, então ele tem menos 2 no ataque e perde a ação de movimento. Dá mais 6. Rolou 14, total 20. Acerta? E já agora, tirem-me uma dúvida, quantos turnos faltam para o feitiço acabar?"

Saída Esperada:
**Ponto de Atrito 1: Gestão temporal de condições e penalizações matemáticas.**
*Visão do Mestre:* Ocorreu uma quebra no ritmo de combate e uma falha na retenção de informação (memory load). O Mestre perdeu o controlo de quantos turnos o feitiço (Lentidão) duraria e teve de recalcular mentalmente os modificadores de ataque do monstro no momento da ação. (Citação: *"espera, ele sofreu aquele feitiço de lentidão, certo?"*)
*Visão do Engenheiro:*
- **RF01 (Monitorização Dinâmica de Estados):** O sistema deve permitir associar estados (condições temporárias) às entidades (NPCs/Monstros), aplicando automaticamente os debuffs matemáticos aos seus atributos.
- **RF02 (Contador de Duração Automático):** O sistema deve decrementar a duração de um estado a cada ciclo de iniciativa e alertar visualmente o Mestre quando o efeito terminar.
- **Justificação de Design (IPM):** Automatizar o rastreio de durações e o recálculo de modificadores liberta a memória de trabalho do Mestre de Jogo, permitindo-lhe focar-se na descrição narrativa da ação em vez da microgestão de regras.
--- FIM DO EXEMPLO ---

Agora, assumindo as mesmas duas personas e aplicando exatamente a mesma estrutura de saída (Ponto de Atrito, Visão do Mestre, Visão do Engenheiro com RFs e Justificação), analise a transcrição seguinte:

Transcrição da Sessão:
{transcricao}
"""

resposta_one = model.generate_content(prompt_one_shot)
print(resposta_one.text)

Esta é uma análise técnica estruturada sobre a sessão de *Tormenta 20* transcrita, focada em otimizar a experiência de jogo e reduzir a carga cognitiva sobre o Mestre e os jogadores.

---

### Ponto de Atrito 1: Gestão de Estado de Combate e "Overhead" Mental
**Visão do Mestre:** Existe uma dificuldade significativa em manter o controle do estado do campo de batalha. O Mestre alterna entre narrador e "computador humano", calculando danos, verificando penalidades de armadura (escudos, penalidade de proficiência) e gerenciando condições de inimigos enquanto tenta descrever a cena. Isso causa interrupções frequentes e incerteza sobre o que é uma ação válida vs. penalidade. (Citação: *"É, isso tá meio travado esse negócio aí... Tu vê que tem a flechinha ali, né? Pra que lado tu pode girar"*).

**Visão do Engenheiro:**
*   **RF01 (Gestão de Equipamento e Status):** O sistema deve registrar o inventário dos personagens e aplicar automaticamente modificadores de penalidade (ex: penalidade por

implementando o RAG (Vector Database)

Para o RAG funcionar, precisamos simular o livro de regras. O código abaixo cria um banco vetorial temporário, insere as regras do Tormenta 20 nele, faz uma busca pela regra mais relevante baseada no que os jogadores falaram e injeta isso no prompt.



In [ ]:
import chromadb
import google.generativeai as genai
from google.colab import userdata

# Configuração do Modelo
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-3.1-flash-lite')

# 2. Lendo o arquivo do Livro de Regras
print("Lendo o livro de regras...")
try:
    with open('regras_tormenta_20.txt', 'r', encoding='utf-8') as f:
        texto_regras = f.read()
except FileNotFoundError:
    print("ERRO: Faça o upload do arquivo 'regras_tormenta_20.txt' na aba Files!")
    texto_regras = ""

# Chunking (Fatiamento de Dados)
# Divide o texto grande em blocos menores baseados em quebras de linha duplas
# Ignora fatias vazias ou muito pequenas (menores que 50 caracteres)
blocos_de_regras = [bloco.strip() for bloco in texto_regras.split('\n\n') if len(bloco.strip()) > 50]

# Configurando o Banco Vetorial (ChromaDB)
chroma_client = chromadb.Client()

# Deleta a coleção se ela já existir para não duplicar dados ao rodar a célula várias vezes
try:
    chroma_client.delete_collection(name="banco_tormenta20")
except Exception: # Catching a more general exception for robustness
    pass

colecao_regras = chroma_client.create_collection(name="banco_tormenta20")

# Criando IDs sequenciais e Metadados para cada bloco fatiado
ids = [f"regra_t20_{i}" for i in range(len(blocos_de_regras))]
metadatas = [{"fonte": f"Bloco_{i}"} for i in range(len(blocos_de_regras))]

print(f"Vetorizando e inserindo {len(blocos_de_regras)} blocos de regras no ChromaDB...")
if blocos_de_regras:
    colecao_regras.add(
        documents=blocos_de_regras,
        metadatas=metadatas,
        ids=ids
    )
else:
    print("AVISO: Nenhum bloco de regras para adicionar ao ChromaDB, verifique o arquivo de regras.")

# 5. Recuperação (Retrieval)
# A transcrição da sessão é usada como "pergunta" (query) para o banco vetorial
# O banco vai devolver os 5 blocos de regras oficiais que mais se assemelham ao que foi conversado na mesa
print("Buscando regras relevantes para a transcrição...")
# Ensure transcricao is not empty before querying
if transcricao and colecao_regras.count() > 0:
    resultados = colecao_regras.query(
        query_texts=[transcricao],
        n_results=5
    )
    # Junta os 5 blocos recuperados em uma única string
    regras_recuperadas = "\n\n--- REGRA ---\n".join(resultados['documents'][0])
else:
    print("AVISO: Nenhuma transcrição para buscar ou banco de regras vazio.")
    regras_recuperadas = "Não foi possível recuperar regras devido à falta de transcrição ou banco de regras vazio."

# Geração Aumentada (Prompt RAG)
prompt_rag = f"""
Assuma um papel duplo para analisar excertos de uma sessão real de RPG de mesa (Tormenta 20) e extrair requisitos de software, utilizando documentos de referência para fundamentar a sua análise.

PERSONA 1: Mestre de Jogo (MJ) veterano de Tormenta 20, com profundo conhecimento das regras e elevada empatia pela sobrecarga cognitiva (memory load) exigida para gerir a mesa.
PERSONA 2: Engenheiro de Requisitos e Investigador de Interação Pessoa-Máquina (IPM), especialista na conceção de sistemas de suporte à decisão para videojogos e TTRPGs.

Abaixo, receberá dois blocos de informação: o [CONTEXTO RECUPERADO] (que contém extratos do livro de regras do Tormenta 20 e/ou literatura académica sobre ferramentas de RPG) e a [TRANSCRIÇÃO DA SESSÃO].

A sua tarefa é cruzar os dados e gerar um relatório estruturado em dois passos:

PASSO 1 (Visão do Mestre): Identifique um "Ponto de Atrito" (Friction Point) na transcrição (ex: confusão com magias, cálculo de bónus, quebra de ritmo).
Cruze este atrito com o [CONTEXTO RECUPERADO] para explicar a mecânica exata que causou o problema ou o conceito teórico de IPM que explica a dificuldade. Cite o excerto da transcrição que comprova o atrito.

PASSO 2 (Visão do Engenheiro): Proponha Requisitos Funcionais (RF) e uma Justificação de Design computacional para mitigar essa dor, baseando-se estritamente nas capacidades que reduziriam a carga cognitiva descrita no Passo 1, sem retirar a agência do Mestre.

Gere a sua resposta seguindo rigorosamente esta estrutura:
- **Ponto de Atrito:** [Título curto do problema]
- **Visão do Mestre (Diagnóstico):** [Análise da situação cruzada com o contexto recuperado. Citação da transcrição]
- **Visão do Engenheiro (Requisitos):**
  - **RF[XX]:** [Nome e descrição funcionalidade]
  - **Justificação de Design:** [Explicação técnica do benefício cognitivo/usabilidade]

==================================================
[CONTEXTO RECUPERADO]
{regras_recuperadas}

[TRANSCRIÇÃO DA SESSÃO]
{transcricao}
"""

print("\nGerando Requisitos com o Gemini...\n")
# Only generate content if regras_recuperadas is not the default warning message
if regras_recuperadas != "Não foi possível recuperar regras devido à falta de transcrição ou banco de regras vazio.":
    resposta_rag = model.generate_content(prompt_rag)
    print("=== REGRAS UTILIZADAS PELO SISTEMA ===")
    print(f"{regras_recuperadas[:300]}... [Texto truncado para exibição]\n")

    print("=== REQUISITOS GERADOS ===")
    print(resposta_rag.text)
else:
    print("Não foi possível gerar requisitos sem regras recuperadas ou transcrição.")

Lendo o livro de regras...
Vetorizando e inserindo 367 blocos de regras no ChromaDB...
Buscando regras relevantes para a transcrição...

Gerando Requisitos com o Gemini...

=== REGRAS UTILIZADAS PELO SISTEMA ===
Independente da aventura que você planejou, ou dos personagens que seus amigos interpretam, você estará lidando com pessoas. Provavelmente eles veem você como uma espécie de líder (já que o mestre tem as maiores responsabilidades). Então, tenha alguns cuidados para que o jogo transcorra bem. Em prim... [Texto truncado para exibição]

=== REQUISITOS GERADOS ===
Esta é uma análise técnica estruturada, combinando a perspectiva da gestão de mesa com a engenharia de sistemas de suporte à decisão.

---

### **Ponto de Atrito: Sobrecarga Cognitiva na Gestão de Combate e Regras**

**Visão do Mestre (Diagnóstico):**
O grupo apresenta uma "paralisia por excesso de informação". A transcrição revela que, durante o combate, o ritmo é constantemente interrompido por dúvidas sobre mecânicas b